### Setting up

In [ ]:
# Import packages
import uuid
import pandas as pd
from os import environ
from auth import Auth
from workspace import Workspace
from dataset import Dataset
from dataflow import Dataflow
from dotenv import load_dotenv

load_dotenv('./utils/.env')

# Tenant/app settings
TENANT_ID = environ.get('TENANT_ID', '')
CLIENT_ID = environ.get('CLIENT_ID', '')
CLIENT_SECRET = environ.get('CLIENT_SECRET', '')

In [ ]:
# Authentication (get bearer token)
auth = Auth(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
token = auth.get_token()

# Initializing objects
workspace = Workspace(token)
dataset = Dataset(token)
dataflow = Dataflow(token)

### Getting workspaces data

In [ ]:
# Get workspaces list
workspaces = workspace.list_workspaces()
workspaces_list = workspaces.get('content', [])

# See content
n = 13
print(f'Found {len(workspaces_list)} workspaces...\nPrinting first {n}:')
df_workspaces = pd.DataFrame(workspaces_list)
df_workspaces.head(n)

### List all users with access to a dataset

In [ ]:
#ataset_users = dataset.list_users()
try:
    workspace_to_search = 'Test workspace'
    workspace_to_search_id = df_workspaces['id'][df_workspaces['name']==workspace_to_search].values[0]
except IndexError:
    workspace_to_search_id = ''
print(f'Workspace ID: {workspace_to_search_id}')

dataset_to_search = 'Test dataset'

if workspace_to_search_id != '':
    datasets_list = dataset.list_datasets(workspace_id=workspace_to_search_id)
    df_datasets = pd.DataFrame(datasets_list['content'])
    dataset_to_search_id = df_datasets['id'][df_datasets['name']==dataset_to_search].values[0]
else:
    dataset_to_search_id = ''
print(f'Dataset ID: {dataset_to_search_id}')

if dataset_to_search_id != '':
    dataset_users = dataset.list_users(workspace_id=workspace_to_search_id, dataset_id=dataset_to_search_id)
    df_dataset_users = pd.DataFrame(dataset_users['content'])
    df_dataset_users['workspace'] = workspace_to_search
    df_dataset_users['report'] = dataset_to_search

    df_dataset_users.to_excel(f'./data/users/{dataset_to_search}_users.xlsx', index=False)

### Add user to all workspaces

In [ ]:
users_to_be_added = [
    {
        'user': 'test@test.com',
        'role': 'Member'
    }
]

for user_to_add in users_to_be_added:

    for workspace_data in workspaces_list:
        workspace_id = workspace_data['id']
        try:
            response = workspace.add_user(user_principal_name=user_to_add['user'], access_right=user_to_add['role'], workspace_id=workspace_id)
            if response['message'] != 'Success':
                response = workspace.update_user(user_principal_name=user_to_add['user'], access_right=user_to_add['role'], workspace_id=workspace_id)
            print(workspace_data['name'], response)
        except Exception as e:
            print(e)

### Remove user from all workspaces

In [ ]:
users_to_be_removed = [
    'test@test.com'
]

for user_to_remove in users_to_be_removed:
    for workspace_data in workspaces_list:
        workspace_id = workspace_data['id']
        try:
            response = workspace.remove_user(user_principal_name=user_to_remove, workspace_id=workspace_id)
            print(workspace_data['name'], response)
        except Exception as e:
            print(e)

### Copy a Dataflow to Another Workspace

In [5]:
source_workspace_id = '123'
destination_workspace_id = '456'
dataflows_list_raw = dataflow.list_dataflows(workspace_id=source_workspace_id)
dataflows_list = dataflows_list_raw['content']

dataflows_to_copy_list = ['dataflow1', 'dataflow2']
dataflows_to_copy = [d for d in dataflows_list if d.get('name', 'not found') in dataflows_to_copy_list]

In [7]:
import requests 
for d in dataflows_to_copy:
    dataflow_id = d['objectId']
    dataflow_name = d['name']

    dataflow_content = dataflow.get_dataflow_details(workspace_id=source_workspace_id, dataflow_id=dataflow_id).get('content', '')
    
    if dataflow_content != '':
        
        dataflow_content['entities'][0].pop('partitions', None)
        dataflow_content['pbi:mashup']['allowNativeQueries'] = False

        r = dataflow.create_dataflow(workspace_id=destination_workspace_id, dataflow_content=dataflow_content)
        print(r)

{'message': 'Success'}
